FEATURE EXTRACTION

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import librosa
import math
from matplotlib.ticker import FormatStrFormatter

In [2]:
# 1. Definizione del percorso del file
CSV_INPUT_PATH = "audio_samples_metadata.csv"
CSV_OUTPUT_PATH = "audio_samples_features.csv"
INPUT_AUDIO_DIR = Path("AudioSamplesPreprocessed")
SPECTROGRAM_DIR = Path("Spectrograms")
SPECTROGRAM_DIR.mkdir(exist_ok=True)

CARICAMENTO DATI

In [3]:
def load_audio_data(csv_input_path):
    audio_df=pd.DataFrame()
    # 2. Caricamento del file CSV nel DataFrame
    if Path(csv_input_path).exists():
        audio_df = pd.read_csv(csv_input_path)
        
        # Conversione della colonna timestamp in oggetti datetime
        audio_df["timestamp"] = pd.to_datetime(audio_df["timestamp"])
        
        print("File CSV caricato correttamente.")
        print(f"Numero di righe lette: {len(audio_df)}")
        
        # Visualizzazione delle prime righe e dei tipi di dati
        print("\nAnteprima del DataFrame:")
        print(audio_df.head())
        print("\nInformazioni sulle colonne:")
        print(audio_df.info())
    else:
        print(f"Errore: Il file {csv_input_path} non esiste nella directory corrente.")

    return audio_df

In [4]:
audiofiles_df = load_audio_data(CSV_INPUT_PATH)

File CSV caricato correttamente.
Numero di righe lette: 2768

Anteprima del DataFrame:
                                   original_filename  \
0  audio_b24926a73b1a38594877bc71e7eaff7aa7eadd2f...   
1  audio_2771e54671ec43392942a62261537b65cbc8695a...   
2  audio_16f20a655c2d3ec352fb5c157b6b976703d5b5da...   
3  audio_121d7740938d3ac5bf777d9c1d64ac738e010942...   
4  audio_9c05e0db805b4407e9ac1c15804a59e8813adc8f...   

                        filename  \
0  audio_2026-02-23T09-24-36.wav   
1  audio_2026-02-23T09-25-47.wav   
2  audio_2026-02-23T09-26-59.wav   
3  audio_2026-02-23T09-28-11.wav   
4  audio_2026-02-23T09-29-22.wav   

                                         sha256_hash  \
0  db54ef4b870968051ea8f8dd6eeae008168fd49cbc9bb4...   
1  8faf173640803ba10210997d2ae765b1f919cac9b28f02...   
2  fa0a909d1b207767b286f47758334a264f0161d78d58db...   
3  5966c07def1ae0a071af3cb9a9594ff1895539dbe71558...   
4  e10257a59c5c09752779981b6b1afe2a123819164df0bd...   

                  time

GENERAL PARAMETERS

In [5]:
SAMPLE_RATE = 16000   #16 kHz
FRAME_SIZE = 2048
HOP_LENGTH = 512
BASE_SPLIT_FREQUENCY = 2000

AMPLITUDE ENVELOPE

In [6]:
def amplitude_envelope(signal, frame_size, hop_length):
    """Calculate the amplitude envelope of a signal with a given frame size and hop length."""
    amplitude_envelope = []
    
    # calculate amplitude envelope for each frame
    for i in range(0, len(signal), hop_length): 
        amplitude_envelope_current_frame = max(signal[i:i+frame_size])
        amplitude_envelope.append(amplitude_envelope_current_frame)
    
    return np.array(amplitude_envelope)

def plot_amplitude_envelope(signal, ae_signal, hop_length, sample_rate):
    frames = range(len(ae_signal))
    t = librosa.frames_to_time(frames, hop_length=hop_length, sr=sample_rate)
    # amplitude envelope is graphed in red

    plt.figure(figsize=(15, 17))

    ax = plt.subplot(3, 1, 1)
    librosa.display.waveshow(signal, alpha=0.5, sr=sample_rate)
    plt.plot(t, ae_signal, color="r")
    plt.ylim((-1, 1))
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.title("Amplitude Envelope")

    plt.show()

In [7]:
# number of frames in amplitude envelope
# ae_audio_signal = amplitude_envelope(audio_signal, FRAME_SIZE, HOP_LENGTH)

#plot_amplitude_envelope(audio_signal, ae_audio_signal, HOP_LENGTH, SAMPLE_RATE)

RMSE

In [8]:
def rmse(signal, frame_size, hop_length):
    return librosa.feature.rms(y=signal, frame_length=frame_size, hop_length=hop_length)[0]

def plot_rmse(signal, rms_signal, hop_length, sample_rate):
    frames = range(len(rms_signal))
    t = librosa.frames_to_time(frames, hop_length=hop_length, sr=sample_rate)
    # rms energy is graphed in red

    plt.figure(figsize=(15, 17))

    ax = plt.subplot(3, 1, 1)
    librosa.display.waveshow(signal, alpha=0.5, sr=sample_rate)
    plt.plot(t, rms_signal, color="r")
    plt.ylim((-1, 1))
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.title("RMS Energy")

    plt.show()

In [9]:
#rms_audio_signal = rmse(audio_signal, FRAME_SIZE, HOP_LENGTH)

#plot_rmse(audio_signal, rms_audio_signal, HOP_LENGTH, SAMPLE_RATE)

ZERO-CROSSING RATE

In [10]:
def zero_crossing_rate(signal, frame_size, hop_length):
    return librosa.feature.zero_crossing_rate(y=signal, frame_length=frame_size, hop_length=hop_length)[0]

def plot_zero_crossing_rate(zcr_signal, hop_length, sample_rate):
    frames=range(len(zcr_signal))
    t = librosa.frames_to_time(frames, hop_length=hop_length, sr=sample_rate)
    plt.figure(figsize=(15, 5))

    plt.plot(t, zcr_signal, color="r")
    plt.xlabel('Time (seconds)')
    plt.ylabel('Zero Crossing Rate')
    plt.ylim(0, 1)
    plt.title("Zero Crossing Rate")
    plt.show()

In [11]:
#zcr_audio_signal = zero_crossing_rate(audio_signal, FRAME_SIZE, HOP_LENGTH)

#plot_zero_crossing_rate(zcr_audio_signal, HOP_LENGTH, SAMPLE_RATE)

FOURIER TRANSFORM

In [12]:
def magnitude_spectrum(signal):
    X = np.fft.fft(signal)
    X_mag = np.absolute(X)
    half = len(X_mag) // 2
    X_mag_half = X_mag[:half]
    return X_mag_half

def plot_magnitude_spectrum(signal, sr, title):
    X_mag = magnitude_spectrum(signal)
    
    plt.figure(figsize=(18, 5))
    
    f = np.linspace(0, sr, len(X_mag)) 
    
    plt.plot(f, X_mag)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.title(title)

In [13]:
#plot_magnitude_spectrum(audio_signal, SAMPLE_RATE, "Audio signal")

SPECTOGRAM

In [14]:
def spectrogram(signal, frame_size, hop_length):
    S_scale = librosa.stft(signal, n_fft=frame_size, hop_length=hop_length)
    Y_scale = np.abs(S_scale) ** 2
    Y_log_scale = librosa.power_to_db(Y_scale)
    return Y_log_scale

def plot_spectrogram(Y, sr, hop_length, y_axis="linear"):
    plt.figure(figsize=(25, 10))
    librosa.display.specshow(Y, 
                             sr=sr, 
                             hop_length=hop_length, 
                             x_axis="time", 
                             y_axis=y_axis)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Mel frequency" if y_axis == "mel" else "Frequency (Hz)")
    if y_axis in ["mel", "log"]:
        formatter = FormatStrFormatter('%+2.0f dB')
    else:
        formatter = FormatStrFormatter('%+2.0f')
    plt.colorbar(format=formatter).set_label("Magnitude", rotation=90, labelpad=15)
    plt.title(f"{'Mel ' if y_axis == 'mel' else ''}Spectrogram")

In [15]:
#audio_signal_spectrogram = spectrogram(audio_signal, FRAME_SIZE, HOP_LENGTH)
#plot_spectrogram(audio_signal_spectrogram, SAMPLE_RATE, HOP_LENGTH, y_axis="log")

MEL SPECTROGRAM

In [16]:
def mel_spectrogram(signal, frame_size, hop_length, sr, n_mels):
    mel_spectrogram = librosa.feature.melspectrogram(y=signal, sr=sr, n_fft=frame_size, hop_length=hop_length, n_mels=n_mels)
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
    return log_mel_spectrogram

'''
filter_banks = librosa.filters.mel(n_fft=FRAME_SIZE, sr=sample_rate, n_mels=10)

plt.figure(figsize=(25, 10))
librosa.display.specshow(filter_banks, 
                         sr=sample_rate, 
                         x_axis="linear")
plt.colorbar(format="%+2.f")
plt.show()
'''

'\nfilter_banks = librosa.filters.mel(n_fft=FRAME_SIZE, sr=sample_rate, n_mels=10)\n\nplt.figure(figsize=(25, 10))\nlibrosa.display.specshow(filter_banks, \n                         sr=sample_rate, \n                         x_axis="linear")\nplt.colorbar(format="%+2.f")\nplt.show()\n'

In [17]:
#audio_signal_mel_spectrogram = mel_spectrogram(audio_signal, FRAME_SIZE, HOP_LENGTH, SAMPLE_RATE, n_mels=10)

#plot_spectrogram(audio_signal_mel_spectrogram, SAMPLE_RATE, HOP_LENGTH, y_axis="mel")

MFCC

In [18]:
def mfcc(signal, sr, n_mfcc):
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=n_mfcc)
    return mfccs

def mfcc_delta(mfccs, order=1):
    delta_mfccs = librosa.feature.delta(mfccs, order=order)
    return delta_mfccs

def plot_mfcc(mfccs, sr, hop_length, ylabel="MFCC", title="MFCC"):
    plt.figure(figsize=(25, 10))
    librosa.display.specshow(mfccs, 
                            x_axis="time", 
                            sr=sr,
                            hop_length=hop_length
                            )
    plt.colorbar(format="%+2.f")
    plt.xlabel('Time (seconds)')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.show()

In [19]:
'''
mfccs = mfcc(audio_signal, SAMPLE_RATE, n_mfcc=13)

delta_mfccs = mfcc_delta(mfccs)

delta2_mfccs = mfcc_delta(mfccs, order=2)

plot_mfcc(mfccs, SAMPLE_RATE, HOP_LENGTH)
plot_mfcc(delta_mfccs, SAMPLE_RATE, HOP_LENGTH, ylabel="Δ MFCC")
plot_mfcc(delta2_mfccs, SAMPLE_RATE, HOP_LENGTH, ylabel="Δ² MFCC")

mfccs_features = np.concatenate((mfccs, delta_mfccs, delta2_mfccs))
'''

'\nmfccs = mfcc(audio_signal, SAMPLE_RATE, n_mfcc=13)\n\ndelta_mfccs = mfcc_delta(mfccs)\n\ndelta2_mfccs = mfcc_delta(mfccs, order=2)\n\nplot_mfcc(mfccs, SAMPLE_RATE, HOP_LENGTH)\nplot_mfcc(delta_mfccs, SAMPLE_RATE, HOP_LENGTH, ylabel="Δ MFCC")\nplot_mfcc(delta2_mfccs, SAMPLE_RATE, HOP_LENGTH, ylabel="Δ² MFCC")\n\nmfccs_features = np.concatenate((mfccs, delta_mfccs, delta2_mfccs))\n'

BAND ENERGY RATIO

In [20]:
def calculate_split_frequency_bin(split_frequency, sample_rate, num_frequency_bins):
    """Infer the frequency bin associated to a given split frequency."""
    
    frequency_range = sample_rate / 2
    frequency_delta_per_bin = frequency_range / num_frequency_bins
    split_frequency_bin = math.floor(split_frequency / frequency_delta_per_bin)
    return int(split_frequency_bin)

def band_energy_ratio(signal,split_frequency, frame_size, hop_length, sample_rate):
    """Calculate band energy ratio with a given split frequency."""

    spectrogram = librosa.stft(signal, n_fft=frame_size, hop_length=hop_length)

    split_frequency_bin = calculate_split_frequency_bin(split_frequency, sample_rate, spectrogram.shape[0])
    band_energy_ratio = []
    
    # calculate power spectrogram
    power_spectrogram = np.abs(spectrogram) ** 2
    power_spectrogram = power_spectrogram.T
    
    # calculate BER value for each frame
    for frame in power_spectrogram:
        sum_power_low_frequencies = frame[:split_frequency_bin].sum()
        sum_power_high_frequencies = frame[split_frequency_bin:].sum()
        band_energy_ratio_current_frame = sum_power_low_frequencies / sum_power_high_frequencies
        band_energy_ratio.append(band_energy_ratio_current_frame)
    
    return np.array(band_energy_ratio)

def plot_band_energy_ratio(ber_signal, hop_length, sample_rate):
    frames = range(len(ber_signal))
    t = librosa.frames_to_time(frames, hop_length=hop_length, sr=sample_rate)
    plt.figure(figsize=(25, 10))

    plt.plot(t, ber_signal, color="b")
    #plt.ylim((0, 4000000))
    plt.xlabel('Time (seconds)')
    plt.ylabel('Band Energy Ratio')
    plt.title("Band Energy Ratio")
    plt.show()


In [21]:
#ber_audio_signal = band_energy_ratio(audio_signal, BASE_SPLIT_FREQUENCY, FRAME_SIZE, HOP_LENGTH, SAMPLE_RATE)

#plot_band_energy_ratio(ber_audio_signal, HOP_LENGTH, SAMPLE_RATE)

SPECTRAL CENTROID

In [22]:
def spectral_centroid(signal, sr, frame_size, hop_length):
    return librosa.feature.spectral_centroid(y=signal, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]

def plot_spectral_centroid(sc_signal, hop_length, sample_rate):
    frames = range(len(sc_signal))
    t = librosa.frames_to_time(frames, hop_length=hop_length, sr=sample_rate)
    plt.figure(figsize=(25,10))

    plt.plot(t, sc_signal, color='b')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Spectral Centroid')
    plt.title("Spectral Centroid")
    plt.show()

In [23]:
#sc_audio_signal = spectral_centroid(audio_signal, SAMPLE_RATE, FRAME_SIZE, HOP_LENGTH)

#plot_spectral_centroid(sc_audio_signal, HOP_LENGTH, SAMPLE_RATE)

SPECTRAL BANDWITH

In [24]:
def spectral_bandwidth(signal, sr, frame_size, hop_length):
    return librosa.feature.spectral_bandwidth(y=signal, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]

def plot_spectral_bandwidth(bandwidth_signal, hop_length, sample_rate):

    frames = range(len(bandwidth_signal))
    t = librosa.frames_to_time(frames, hop_length=hop_length, sr=sample_rate)
    plt.figure(figsize=(25,10))

    plt.plot(t, bandwidth_signal, color='b')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Spectral Bandwidth')
    plt.title("Spectral Bandwidth")
    plt.show()

In [25]:
#bandwidth_audio_signal = spectral_bandwidth(audio_signal, SAMPLE_RATE, FRAME_SIZE, HOP_LENGTH)

#plot_spectral_bandwidth(bandwidth_audio_signal, HOP_LENGTH, SAMPLE_RATE)

FEATURE EXTRACTION

In [26]:
def extract_stats(array):

    array = np.ravel(array)
    
    mean = np.mean(array)
    std = np.std(array)
    q1 = np.percentile(array, 25)
    median = np.percentile(array, 50)  # q2
    q3 = np.percentile(array, 75)
    iqr = q3 - q1
    delta_mean = np.mean(np.diff(array))
    min = np.min(array)
    max = np.max(array)
    
    return {
        'mean': mean,
        'std': std,
        'min': min,
        'max': max,
        'q1': q1,
        'median': median,
        'q3': q3,
        'iqr': iqr,
        'delta_mean':  delta_mean
    }

In [27]:
all_audio_data = []

for i, row in audiofiles_df.iterrows():

    filename = row["filename"]


    file_path = INPUT_AUDIO_DIR / filename

    
    audio_signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    #print(f"File caricato: {file_path}")
    
    ae = amplitude_envelope(audio_signal, FRAME_SIZE, HOP_LENGTH)
    rms = rmse(audio_signal, FRAME_SIZE, HOP_LENGTH)
    zcr = zero_crossing_rate(audio_signal, FRAME_SIZE, HOP_LENGTH)
    mel= mel_spectrogram(audio_signal, FRAME_SIZE, HOP_LENGTH, SAMPLE_RATE, n_mels=10)
    ber = band_energy_ratio(audio_signal, BASE_SPLIT_FREQUENCY, FRAME_SIZE, HOP_LENGTH, SAMPLE_RATE)
    sc = spectral_centroid(audio_signal, SAMPLE_RATE, FRAME_SIZE, HOP_LENGTH)
    bw = spectral_bandwidth(audio_signal, SAMPLE_RATE, FRAME_SIZE, HOP_LENGTH)

    # --- Costruzione della riga per il DataFrame ---
    current_file_features = {'filename': row.filename}

    # 1. Processiamo le feature standard (1D)
    features_1d = {
        "ae": ae, "rms": rms, "zcr": zcr, 
        "ber": ber, "sc": sc, "bw": bw
    }

    for name, signal in features_1d.items():
        stats = extract_stats(signal)
        for stat_name, value in stats.items():
            current_file_features[f"{name}_{stat_name}"] = value

    # 2. Processiamo il Mel Spectrogram (10 bande)
    # Calcoliamo le statistiche per OGNI banda di frequenza
    # mel ha shape (10, frames), quindi iteriamo sulle 10 righe
    for m in range(mel.shape[0]):
        band_stats = extract_stats(mel[m, :])
        for stat_name, value in band_stats.items():
            current_file_features[f"mel_b{m}_{stat_name}"] = value

    # Aggiungiamo la riga alla lista
    all_audio_data.append(current_file_features)
    print(f"[{i}/{len(audiofiles_df)}] Feature estratte per: {row.filename}")

# --- Creazione del DataFrame finale ---
features_df = pd.DataFrame(all_audio_data)


c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-24-36.wav
[0/2768] Feature estratte per: audio_2026-02-23T09-24-36.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-25-47.wav
[1/2768] Feature estratte per: audio_2026-02-23T09-25-47.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-26-59.wav
[2/2768] Feature estratte per: audio_2026-02-23T09-26-59.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-28-11.wav
[3/2768] Feature estratte per: audio_2026-02-23T09-28-11.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-29-22.wav


c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\numpy\_core\_methods.py:190: RuntimeWarning: overflow encountered in square
  x = um.square(x, out=x)


[4/2768] Feature estratte per: audio_2026-02-23T09-29-22.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-34-12.wav
[5/2768] Feature estratte per: audio_2026-02-23T09-34-12.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-39-01.wav
[6/2768] Feature estratte per: audio_2026-02-23T09-39-01.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-43-43.wav
[7/2768] Feature estratte per: audio_2026-02-23T09-43-43.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-48-33.wav
[8/2768] Feature estratte per: audio_2026-02-23T09-48-33.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T09-55-18.wav
[9/2768] Feature estratte per: audio_2026-02-23T09-55-18.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T10-00-09.wav
[10/2768] Feature estratte per: audio_2026-02-23T10-00-09.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T10-04-57.wav
[11/2768] Feature estratte per: audio_2026-02-23T10-04-57.wav
File caricato: Audio

C:\Users\chris\AppData\Local\Temp\ipykernel_12168\3890383901.py:25: RuntimeWarning: invalid value encountered in scalar divide
  band_energy_ratio_current_frame = sum_power_low_frequencies / sum_power_high_frequencies


[69/2768] Feature estratte per: audio_2026-02-23T14-21-04.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-22-16.wav
[70/2768] Feature estratte per: audio_2026-02-23T14-22-16.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-23-27.wav
[71/2768] Feature estratte per: audio_2026-02-23T14-23-27.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-24-39.wav
[72/2768] Feature estratte per: audio_2026-02-23T14-24-39.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-25-50.wav
[73/2768] Feature estratte per: audio_2026-02-23T14-25-50.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-38-21.wav
[74/2768] Feature estratte per: audio_2026-02-23T14-38-21.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-39-33.wav
[75/2768] Feature estratte per: audio_2026-02-23T14-39-33.wav
File caricato: AudioSamplesPreprocessed\audio_2026-02-23T14-40-45.wav
[76/2768] Feature estratte per: audio_2026-02-23T14-40-45.wav
File caricato:

In [28]:
features_df.to_csv(CSV_OUTPUT_PATH, index=False)